# Advanced Weather Data Analysis with Pandas
### Data Cleaning • Exploratory Data Analysis • Statistical Comparisons • Visualizations

**Author:** Harpreet Singh  
**Updated:** October 2026  
**Environment:** Python 3 • Jupyter Notebook / Google Colab

> This upgraded, self-contained educational project supports an uploaded weather CSV. If none is available, it creates a **clearly labeled synthetic demonstration dataset**. Synthetic results are not real weather observations.

**Portfolio project:** Interactive dashboard: `dashboard/app.py` · Dependencies: `requirements.txt` · Author: **Harpreet Singh**


## 1. Setup and configuration
Run cells from top to bottom. Install missing packages with `%pip install pandas numpy matplotlib seaborn` if needed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (11, 5), "figure.dpi": 110})
RANDOM_SEED = 42
DATA_PATH = Path("Sample_Weather_Data.csv")  # Change to your own CSV path
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Pandas {pd.__version__} | NumPy {np.__version__}")

## 2. Load a real CSV or generate demonstration data
The original notebook required a private Google Drive file. This version works without that path. Set `DATA_PATH` to use real observations.

In [ ]:
if DATA_PATH.is_file():
    df = pd.read_csv(DATA_PATH)
    data_source = f"Real CSV: {DATA_PATH}"
else:
    rng = np.random.default_rng(RANDOM_SEED)
    timestamps = pd.date_range("2025-01-01", periods=24 * 365, freq="h")
    seasonality = 12 * np.sin(2 * np.pi * (timestamps.dayofyear.to_numpy() - 80) / 365)
    temperature = 12 + seasonality + rng.normal(0, 4, len(timestamps))
    humidity = np.clip(70 - 0.7 * temperature + rng.normal(0, 13, len(timestamps)), 15, 100)
    wind = np.clip(rng.gamma(2.2, 5, len(timestamps)), 0, 75)
    conditions = np.where(humidity > 83, "Rain", np.where(humidity > 60, "Cloudy", "Clear"))
    df = pd.DataFrame({
        "Date/Time": timestamps, "Temp_C": temperature.round(1),
        "Rel Hum_%": humidity.round(0), "Wind Speed_km/h": wind.round(1),
        "Visibility_km": np.clip(20 - humidity / 9 + rng.normal(0, 2, len(timestamps)), 0.1, 25).round(1),
        "Press_kPa": rng.normal(101.3, 0.8, len(timestamps)).round(2),
        "Weather": conditions,
    })
    data_source = "SYNTHETIC demonstration data (not actual weather records)"
print("Data source:", data_source)
display(df.head())

## 3. Validate and standardize columns

In [ ]:
df = df.copy()
df.columns = df.columns.str.strip()
required = {"Date/Time", "Weather", "Wind Speed_km/h"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"Missing required CSV columns: {sorted(missing)}. Found: {df.columns.tolist()}")
df["Date/Time"] = pd.to_datetime(df["Date/Time"], errors="coerce")
for col in ["Wind Speed_km/h", "Temp_C", "Rel Hum_%", "Visibility_km", "Press_kPa"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
df["Weather"] = df["Weather"].astype("string").str.strip()
df = df.dropna(subset=["Date/Time"]).drop_duplicates().sort_values("Date/Time").reset_index(drop=True)
df["Month"] = df["Date/Time"].dt.month
df["Hour"] = df["Date/Time"].dt.hour
print(f"Validated {len(df):,} rows and {len(df.columns)} columns")

## 4. Dataset overview and quality audit

In [ ]:
print("Shape:", df.shape, "| Size:", df.size, "| Dimensions:", df.ndim)
print("Date range:", df["Date/Time"].min(), "to", df["Date/Time"].max())
display(df.head(8), df.tail(3))
df.info()

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(2),
    "unique": df.nunique(dropna=True),
})
display(quality.sort_values("missing_pct", ascending=False))
print("Duplicate rows remaining:", df.duplicated().sum())

## 5. Descriptive statistics and categorical frequencies

In [ ]:
display(df.describe(include="all").T)
weather_counts = df["Weather"].value_counts(dropna=False)
display(weather_counts.rename_axis("Weather").reset_index(name="Count"))
print("Unique weather labels:", df["Weather"].nunique())
print("Non-null wind observations:", df["Wind Speed_km/h"].count())

## 6. Original notebook questions — corrected and dynamic
Unlike the original notebook, results are calculated from the loaded data rather than assuming a fixed number of records.

In [ ]:
clear_count = int(df["Weather"].eq("Clear").sum())
wind_4 = df.loc[np.isclose(df["Wind Speed_km/h"], 4, atol=1e-9, equal_nan=False),
                ["Date/Time", "Weather", "Wind Speed_km/h"]]
object_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()
print("Clear observations:", clear_count)
print("Wind speed exactly 4 km/h:", len(wind_4))
print("Text columns:", object_columns)
display(wind_4.head(10))

## 7. Chart: weather condition comparison

In [ ]:
fig, ax = plt.subplots()
weather_counts.head(12).sort_values().plot.barh(ax=ax, color=sns.color_palette("viridis", min(12, len(weather_counts))))
ax.set(title="Most common weather conditions", xlabel="Number of observations", ylabel="Weather")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "weather_conditions.png", bbox_inches="tight")
plt.show()

## 8. Chart: numeric distributions

In [ ]:
numeric_cols = [c for c in ["Temp_C", "Rel Hum_%", "Wind Speed_km/h", "Visibility_km", "Press_kPa"] if c in df]
if numeric_cols:
    axes = df[numeric_cols].hist(bins=30, figsize=(13, 3 * ((len(numeric_cols) + 1) // 2)), layout=((len(numeric_cols) + 1) // 2, 2))
    plt.suptitle("Distribution of numeric weather measurements", y=1.01)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "numeric_distributions.png", bbox_inches="tight")
    plt.show()

## 9. Chart: monthly trend comparison

In [ ]:
monthly = df.groupby("Month")[numeric_cols].mean(numeric_only=True)
trend_cols = [c for c in ["Temp_C", "Wind Speed_km/h"] if c in monthly]
if trend_cols:
    ax = monthly[trend_cols].plot(marker="o", figsize=(11, 5))
    ax.set(title="Monthly averages (different units; compare trends carefully)", xlabel="Month", ylabel="Average measurement")
    ax.set_xticks(range(1, 13))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "monthly_trends.png", bbox_inches="tight")
    plt.show()
display(monthly.round(2))

## 10. Chart: correlation heatmap
Correlation measures linear association; it does not establish causation.

In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
    ax.set_title("Correlation between numeric weather features")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "correlation_heatmap.png", bbox_inches="tight")
    plt.show()

## 11. Chart: weather categories vs wind speed

In [ ]:
top_conditions = df["Weather"].value_counts().head(6).index
subset = df[df["Weather"].isin(top_conditions)].dropna(subset=["Wind Speed_km/h"])
if not subset.empty:
    fig, ax = plt.subplots(figsize=(11, 5))
    sns.boxplot(data=subset, x="Weather", y="Wind Speed_km/h", ax=ax, color="skyblue")
    ax.set_title("Wind speed distribution by weather condition")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "wind_by_weather.png", bbox_inches="tight")
    plt.show()

## 12. Hourly pattern and comparison table

In [ ]:
hourly = df.groupby("Hour")[numeric_cols].mean(numeric_only=True)
if "Temp_C" in hourly:
    fig, ax = plt.subplots()
    hourly["Temp_C"].plot(ax=ax, marker=".")
    ax.set(title="Average temperature by hour", xlabel="Hour of day", ylabel="Temperature (°C)")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "hourly_temperature.png", bbox_inches="tight")
    plt.show()
comparison = df.groupby("Weather").agg(
    observations=("Weather", "size"),
    mean_wind_kmh=("Wind Speed_km/h", "mean"),
    median_wind_kmh=("Wind Speed_km/h", "median")
).sort_values("observations", ascending=False)
display(comparison.round(2))

## 13. Export clean data, summary tables and charts

In [ ]:
df.to_csv(OUTPUT_DIR / "cleaned_weather_data.csv", index=False)
quality.to_csv(OUTPUT_DIR / "data_quality_report.csv")
comparison.to_csv(OUTPUT_DIR / "weather_comparison.csv")
monthly.to_csv(OUTPUT_DIR / "monthly_averages.csv")
print("Saved outputs to:", OUTPUT_DIR.resolve())
print("Files:", *sorted(p.name for p in OUTPUT_DIR.iterdir()), sep="\n - ")

## 14. Key interpretation guidelines
- Check the **data source** before making claims: synthetic demonstration values are not real-world observations.
- Missing values, sample size, seasonality, and inconsistent units can affect conclusions.
- A correlation heatmap does not prove a cause-and-effect relationship.
- Use actual CSV data and rerun all cells to generate valid findings for your project.

---
**Project author: Harpreet Singh**  
**Notebook:** Advanced Weather Data Analysis with Pandas